## Step 4  
Inputs:  
- s3://thesis--ec331-s3/melted-price-bids/
- s3://thesis--ec331-s3/melted-volume-bids/  

Output: s3://thesis--ec331-s3/merged-price-volume-bids/  

Problems:
- I don't think that there is Settlement date in volume bids, I believe it is now trading bids

In [ ]:
# Melted volume
import pandas as pd

# Define the file path
file_path = 's3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/enriched_volume_bids_20250312_102609_chunk1000of2123_part0001.parquet'

# Read the CSV file into a DataFrame, skipping the first row
volume_df = pd.read_parquet(file_path)

# Display general information about the DataFrame
print("DataFrame Info:")
volume_df.info()

# Show the first few rows of the DataFrame
volume_df.head()

# Show the last few rows of the DataFrame
print("\nDataFrame Tail:")
print(volume_df.tail())

# Display descriptive statistics for numerical columns
print("\nDataFrame Description:")
print(volume_df.describe(include='all'))

# List all column names
print("\nDataFrame Columns:")
print(volume_df.columns.tolist())

In [ ]:
volume_df.head(5)

In [ ]:
volume_df.dtypes

In [ ]:
df[

In [ ]:
# Melted volume
import pandas as pd

# Define the file path
file_path = 's3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_FILTERED_202310010000/raise1sec_bids_melted_20250315175551_part0001.parquet'

# Read the CSV file into a DataFrame, skipping the first row
df = pd.read_parquet(file_path)

# Display general information about the DataFrame
print("DataFrame Info:")
df.info()

# Show the first few rows of the DataFrame
df.head()

# Show the last few rows of the DataFrame
print("\nDataFrame Tail:")
print(df.tail())

# Display descriptive statistics for numerical columns
print("\nDataFrame Description:")
print(df.describe(include='all'))

# List all column names
print("\nDataFrame Columns:")
print(df.columns.tolist())

In [1]:
import awswrangler as wr
import pandas as pd

def main():
    # 1. Define your S3 paths (wildcard-like directory approach).
    volume_path = "s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"
    price_path = "s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_FILTERED_202310010000/"
    output_path = "s3://thesis--ec331-s3/merged-price-volume-bids/"

    # 2. Read both datasets as Pandas DataFrames via awswrangler
    #    (if these directories were written in a "dataset" style, set dataset=True)
    volume_df = wr.s3.read_parquet(path=volume_path, dataset=True)
    price_df = wr.s3.read_parquet(path=price_path, dataset=True)
    
    print("Volume columns:", volume_df.columns)
    print("Price columns:", price_df.columns)

    # 3. Convert date columns to datetime as needed
    volume_df["TRADINGDATE"] = pd.to_datetime(volume_df["TRADINGDATE"], errors="coerce")
    price_df["SETTLEMENTDATE"] = pd.to_datetime(price_df["SETTLEMENTDATE"], errors="coerce")

    # 4. Merge on different column names: 
    #    "TRADINGDATE" in volume_df vs. "SETTLEMENTDATE" in price_df
    merged_df = volume_df.merge(
        price_df,
        left_on=["TRADINGDATE", "DUID", "BIDBAND"],
        right_on=["SETTLEMENTDATE", "DUID", "BIDBAND"],
        how="left",
        suffixes=("_volume", "_price")
    )

    # 5. Write merged result to Parquet in S3
    #    Setting "dataset=True" if you want partitioning; below is the simplest form:
    wr.s3.to_parquet(
        df=merged_df,
        path=output_path,
        index=False,
        dataset=True  # or False if you prefer a single file; dataset=True is typical in data-lake style
    )

    print(f"Done! Merged data is written to {output_path}")

if __name__ == "__main__":
    main()

KeyboardInterrupt: 